# SASA-IT Multi-Model AI API Integration

Dieses Notebook zeigt, wie verschiedene KI-Modelle über ihre APIs angesprochen werden können:

| Anbieter | Modell | Zugang | Zweck |
|----------|--------|--------|-------|
| **NVIDIA NIM** | StarCoder2, DeepSeek, weitere | API Key vorhanden | Code, Text, Reasoning |
| **Ollama** | Llama, Qwen, Gemma etc. | Cloud API | Lokal/Cloud Integration |
| **OpenAI** | GPT-4, Codex | Free Tier | Allzweck, Code |
| **Anthropic** | Claude | Free Tier | Analyse, Text |
| **Qwen** | Qwen Code | API | Code-Generierung |

🔑 **Konfiguration**: Alle Keys aus `.env` (nicht im Code!)

In [ ]:
# === Konfiguration laden ===
import os
from dotenv import load_dotenv

# Lade .env-Datei (im Projekt-Root)
load_dotenv()

NIM_API_KEY = os.getenv('NIM_API_KEY')
NIM_BASE_URL = os.getenv('NIM_BASE_URL', 'https://integrate.api.nvidia.com/v1')

print('✓ NVIDIA NIM konfiguriert' if NIM_API_KEY else '✗ NIM Key fehlt!')
print(f'  Endpoint: {NIM_BASE_URL}')

---
## 1. NVIDIA NIM API (OpenAI-kompatibel)

NIM bietet eine OpenAI-kompatible Endpoint-Schnittstelle. Das ist der einfachste Weg.

**Verfügbare Modelle auf NIM**:
- `bigcode/starcoder2-15b` — Code-Generierung
- `deepseek-ai/deepseek-coder-6.7b-instruct` — Code + Reasoning
- `deepseek-ai/deepseek-v4-flash-0731` — Schnell, leicht
- `deepseek-ai/deepseek-v4-pro-0813` — Starkes Reasoning
- Viele weitere (84 Modelle verfügbar)

In [ ]:
from openai import OpenAI

# NIM als OpenAI-Client konfigurieren
nim_client = OpenAI(
    api_key=NIM_API_KEY,
    base_url=NIM_BASE_URL
)

def nim_generate(prompt: str, model: str = 'bigcode/starcoder2-15b', max_tokens: int = 2000) -> str:
    '''Code oder Text mit NVIDIA NIM generieren'''
    response = nim_client.chat.completions.create(
        model=model,
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=max_tokens,
        temperature=0.2
    )
    return response.choices[0].message.content

# Beispiel: Python-Code generieren
test_code = nim_generate(
    'Schreibe ein kurzes Python-Snippet das die Fibonacci-Folge bis n berechnet',
    model='bigcode/starcoder2-15b'
)
print(test_code)

---
## 2. Ollama Cloud API

Wenn Ollama Cloud verfügbar ist, kannst du es so nutzen:

In [ ]:
# Ollama Cloud Endpoint (falls verfügbar)
# Placeholder — später mit echtem Endpoint befüllen

OLLAMA_API_KEY = os.getenv('OLLAMA_API_KEY', '')
OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434')

def ollama_generate(prompt: str, model: str = 'llama3.2') -> str:
    '''Generiere mit Ollama (lokal oder Cloud)'''
    import requests
    resp = requests.post(
        f'{OLLAMA_BASE_URL}/api/generate',
        json={
            'model': model,
            'prompt': prompt,
            'stream': False
        },
        headers={'Authorization': f'Bearer {OLLAMA_API_KEY}'} if OLLAMA_API_KEY else {}
    )
    return resp.json().get('response', '')

print('Ollama-Funktion definiert. API-Key konfiguriert:', bool(OLLAMA_API_KEY))

---
## 3. OpenAI / GPT (Free Tier)

Für GPT-4 und Codex mit Free Tier:

In [ ]:
# OpenAI API Key (Free Tier)
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', '')

def gpt_generate(prompt: str, model: str = 'gpt-4o-mini', max_tokens: int = 1000) -> str:
    '''GPT-Modell anfragen'''
    if not OPENAI_API_KEY:
        return '⚠ Kein OpenAI API Key konfiguriert'
    client = OpenAI(api_key=OPENAI_API_KEY)
    response = client.chat.completions.create(
        model=model,
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=max_tokens
    )
    return response.choices[0].message.content

print('GPT-Funktion bereit. Key vorhanden:', bool(OPENAI_API_KEY))

---
## 4. Anthropic / Claude (Free Tier)

In [ ]:
from anthropic import Anthropic

ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY', '')

def claude_generate(prompt: str, model: str = 'claude-3-5-sonnet-20241022') -> str:
    '''Claude-Anfrage'''
    if not ANTHROPIC_API_KEY:
        return '⚠ Kein Anthropic API Key konfiguriert'
    client = Anthropic(api_key=ANTHROPIC_API_KEY)
    message = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=[{'role': 'user', 'content': prompt}]
    )
    return message.content[0].text

print('Claude-Funktion bereit. Key vorhanden:', bool(ANTHROPIC_API_KEY))

---
## 5. Qwen Code API

In [ ]:
QWEN_API_KEY = os.getenv('QWEN_API_KEY', '')
QWEN_BASE_URL = os.getenv('QWEN_BASE_URL', '')

def qwen_generate(prompt: str, model: str = 'qwen-max') -> str:
    '''Qwen Code API Anfrage'''
    if not QWEN_API_KEY:
        return '⚠ Kein Qwen API Key konfiguriert'
    import requests
    resp = requests.post(
        f'{QWEN_BASE_URL}/chat/completions',
        headers={
            'Authorization': f'Bearer {QWEN_API_KEY}',
            'Content-Type': 'application/json'
        },
        json={
            'model': model,
            'messages': [{'role': 'user', 'content': prompt}],
            'max_tokens': 2000
        }
    )
    return resp.json().get('choices', [{}])[0].get('message', {}).get('content', '')

print('Qwen-Funktion bereit. Key vorhanden:', bool(QWEN_API_KEY))

---
## 6. Multi-Model Router

Ein intelligenter Router wählt das beste Modell für die Aufgabe:

In [ ]:
class MultiModelRouter:
    '''Router für verschiedene KI-Modelle je nach Aufgabe'''
    
    def __init__(self):
        self.models = {
            'code': [
                ('NIM-StarCoder2', lambda p: nim_generate(p, 'bigcode/starcoder2-15b')),
                ('NIM-DeepSeek', lambda p: nim_generate(p, 'deepseek-ai/deepseek-coder-6.7b-instruct')),
                ('GPT-Codex', lambda p: gpt_generate(p, 'gpt-4o-mini')),
                ('Qwen', lambda p: qwen_generate(p, 'qwen-max')),
            ],
            'reasoning': [
                ('NIM-DeepSeek-Pro', lambda p: nim_generate(p, 'deepseek-ai/deepseek-v4-pro-0813')),
                ('Claude', lambda p: claude_generate(p)),
                ('GPT', lambda p: gpt_generate(p, 'gpt-4o-mini')),
            ],
            'text': [
                ('Claude', lambda p: claude_generate(p)),
                ('GPT', lambda p: gpt_generate(p, 'gpt-4o-mini')),
            ],
            'wiki': [
                ('NIM-StarCoder2', lambda p: nim_generate(p, 'bigcode/starcoder2-15b')),
                ('Claude', lambda p: claude_generate(p)),
            ],
        }
    
    def generate(self, prompt: str, task_type: str = 'code', model_preference: str = None) -> str:
        '''Generiere mit dem besten Modell für die Aufgabe'''
        if task_type not in self.models:
            task_type = 'code'
        
        if model_preference:
            for name, func in self.models[task_type]:
                if model_preference.lower() in name.lower():
                    return func(prompt)
        
        # Standard: Erstes verfügbares Modell
        for name, func in self.models[task_type]:
            result = func(prompt)
            if not result.startswith('⚠'):
                return result
        
        return 'Kein Modell verfügbar'

# Test des Routers
router = MultiModelRouter()
print('Multi-Model Router initialisiert')
print('Verfügbare Aufgaben: code, reasoning, text, wiki')

---
## 7. Beispiel: Wiki-Inhalt mit KI generieren

Wie KI beim Wiki auf sasa-it.de/wiki helfen kann:

In [ ]:
# Wiki-Inhalt mit KI generieren
wiki_prompt = '''Erstelle einen kurzen Wiki-Artikel über "Facility Management" im Stil von SASA-IT.
Inhalte:
- Definition: Was ist Facility Management?
- Dienstleistungen: Gebäudeautomation, Wartung, Energie-Management
- Technologie: IoT, Smart Building, Überwachungssysteme
- Vorteile für Kunden
Format: Markdown, mit Überschriften, Listen, und einem kurzen Einleitungsabsatz.'''

wiki_content = router.generate(wiki_prompt, task_type='wiki')
print(wiki_content)

---
## Zusammenfassung

Dieses Notebook zeigt die Integration aller verfügbaren KI-Modelle:
- **NVIDIA NIM** (StarCoder2, DeepSeek) — primär für Code & Reasoning
- **GPT/Codex** — Allzweck, Free Tier
- **Claude** — Text, Analyse, Wiki-Inhalte
- **Qwen** — Code-Generierung
- **Ollama** — Cloud/Lokal-Integration

Der **MultiModelRouter** wählt automatisch das beste Modell für jede Aufgabe.

📁 Speichere Keys in `.env`, NICHT im Code!